# Storm catalog post-processing and QC

This notebook performs final quality-control and post-processing on the EarthCARE lightning storm catalog.

It:
- parses processing logs to identify missing LI/GLM time intervals
- flags clusters (storms) affected by missing temporal coverage
- masks missing minute bins in `minute_counts`
- removes manually identified bad clusters
- exports the final curated JSON catalog

### Inputs and configuration

This notebook uses:
- an input storm catalog JSON
- processing log files
- public GOES GLM data on S3 for gap detection

In [ ]:
import pandas as pd
from pathlib import Path
import json

from storm_catalogue_qc import (
    extract_missing_intervals_from_logs,
    add_missing_peak_minutes,
    mask_missing_minute_counts,
    json_safe,
)

In [ ]:
LOG_DIR = Path("/path/to/logs")
INPUT_CATALOG_PATH = Path("/path/to/EarthCARE_lightning_storm_catalogue_input.json")
OUTPUT_CATALOG_PATH = Path("/path/to/EarthCARE_lightning_storm_catalogue_final.json")

### Load input storm catalog and log files

In [ ]:
with open(INPUT_CATALOG_PATH, "r") as f:
    catalog_payload = json.load(f)

field_descriptions = catalog_payload["metadata"]
records = catalog_payload["data"]
summary_info = catalog_payload["summary"]

storm_df = pd.DataFrame(records)
storm_df["peak_datetime"] = pd.to_datetime(storm_df["peak_datetime"], utc=True)

log_files = sorted(LOG_DIR.glob("202*.log"))

### Compute cluster-level missing-data diagnostics

Two diagnostics are added:
- `missing_peak_minutes`: overlap of missing data with ±2.5 minutes around peak time
- `masked_minute_counts`: minute-level counts with missing windows masked as null

In [ ]:
missing_intervals_df = extract_missing_intervals_from_logs(log_files, storm_df)
missing_intervals_df["missing_start"] = pd.to_datetime(missing_intervals_df["missing_start"], utc=True)
missing_intervals_df["missing_end"] = pd.to_datetime(missing_intervals_df["missing_end"], utc=True)

storm_catalog_with_missing_peak = add_missing_peak_minutes(storm_df, missing_intervals_df)
storm_catalog_with_masked_counts = mask_missing_minute_counts(storm_catalog_with_missing_peak, missing_intervals_df)

### Apply manual quality-control exclusions
This section removes clusters identified as unreliable during manual quality control, including:
- LI clusters near the edge of the field of view, which may be affected by parallax
- clusters appearing twice at the boundaries of neighboring EarthCARE frames
- other extreme or spurious clusters identified by comparison with EarthCARE radar profiles, for example storm-edge detections without a clear radar signal

In [ ]:
MANUALLY_EXCLUDED_STORM_IDS = ['01019D_LI_0', '01019D_LI_2', '01020D_LI_0', '01189A_GLM_2', '01300D_LI_6', '01386B_LI_1', '01490E_LI_1', '01538D_GLM_7', '01544H_LI_5', '01554E_GLM_1', '01848E_LI_16', '02019E_LI_2', '02191E_GLM_33', '02221E_LI_5', '02221E_LI_6', '02221E_LI_21', '02221E_LI_29', '02299E_LI_2', '02299E_LI_6', '02299E_LI_7', '02299E_LI_8', '02306H_LI_3', '02306H_GLM_2', '02408E_LI_13', '02579E_LI_9', '02579E_LI_25', '02591E_LI_3', '02610E_GLM_13', '02657E_LI_18', '02836H_GLM_19', '02875F_GLM_3', '02913H_LI_5', '02914A_GLM_0', '02914A_LI_0', '02967D_LI_1', '02967D_LI_4', '02967D_GLM_2', '03077E_LI_4', '03077E_LI_5', '03077E_LI_8', '03077E_LI_10', '03310F_GLM_1', '03388E_LI_2', '03403D_LI_0', '03746E_LI_5', '03770A_LI_2', '04166E_LI_1', '04166E_LI_21', '04166E_LI_28', '04166E_LI_29', '04291E_GLM_7', '04291F_GLM_5', '04306E_LI_7', '04508F_LI_0', '04517A_LI_1', '04555E_LI_0', '04751A_GLM_15', '05015A_LI_2', '05015A_LI_3', '05072B_LI_0', '05382E_GLM_0', '05461A_LI_1', '05517A_GLM_1', '05592B_GLM_3', '05708E_GLM_1', '05874D_LI_2', '05874D_LI_6', '05879D_GLM_0', '05907D_LI_1', '06138D_LI_6', '06173E_GLM_1', '06216D_LI_0', '06216D_LI_2', '06244A_LI_0', '06244A_LI_1', '06268D_GLM_4', '06329E_LI_0', '06356D_LI_5', '06356D_LI_6', '06404D_LI_5', '06529D_LI_3', '06609F_GLM_1', '06627D_GLM_0', '06627D_GLM_6', '06653D_LI_1', '06673D_GLM_16', '06745D_LI_1', '06784B_LI_1', '06816B_LI_1', '07006A_LI_7', '07015D_GLM_1', '07107E_LI_1', '07124E_GLM_16', '07209B_GLM_2', '07216E_LI_3', '07257A_GLM_3', '07426H_GLM_32', '07496E_LI_10', '07496E_LI_11', '07574E_LI_11', '07636E_LI_0', '07636E_LI_1', '07636E_LI_4', '07660A_LI_3', '07738B_GLM_1', '07799H_LI_3', '07799H_GLM_18', '07887E_GLM_0', '07940A_LI_2', '08103E_LI_17', '08134E_LI_8', '08134E_LI_9', '08267A_LI_1', '08274E_LI_0', '08348F_LI_4', '08399E_GLM_4', '08553F_LI_2', '08663E_LI_8', '08663E_LI_9', '08663E_LI_17', '08663E_LI_19', '08749A_GLM_4', '08765A_LI_7', '08828A_GLM_1', '08908F_LI_3', '08990E_GLM_6', '08990F_GLM_17', '09122A_LI_5', '09122A_LI_13', '09122A_LI_16', '09122A_LI_19', '09122A_LI_20', '09157E_LI_0', '09223E_GLM_0', '09251E_LI_7', '09251E_LI_17', '09270E_LI_3', '09301E_LI_12', '09301E_LI_38', '09472E_LI_2', '09472E_LI_5']

In [ ]:
exclude_mask = (
    ((storm_catalog_with_masked_counts["source"] == "LI") &
        (
            (storm_catalog_with_masked_counts["peak_lat"].abs() >= 60) |
            (storm_catalog_with_masked_counts["peak_lon"].abs() >= 60)
        ))
    |
    (storm_catalog_with_masked_counts["unique_id"].isin(MANUALLY_EXCLUDED_STORM_IDS))
)

final_storm_catalog_df = storm_catalog_with_masked_counts[~exclude_mask].reset_index(drop=True)

### Export final cleaned catalogue

The final cleaned storm catalogue is exported as JSON with updated metadata.

In [ ]:
field_descriptions["missing_peak_minutes"] = (
    "Total minutes of missing data overlapping ±2.5 min around peak_datetime"
)

field_descriptions["masked_minute_counts"] = (
    "Lightning group counts per minute (–60…+60) with missing intervals masked as null"
)

output = {
    "summary": summary_info,
    "metadata": field_descriptions,
    "data": final_storm_catalog_df.to_dict(orient="records")
}

with open(OUTPUT_CATALOG_PATH, "w") as f:
    json.dump(output, f, indent=2, default=json_safe)